# **Deskripsi masalah**

**Masalah.** Memperkirakan **total beban listrik (VA) lima peralatan bersama**  (mesin kopi, kulkas, ketel, microwave, printer) pada setiap interval 11,4 jam,
berdasarkan (a) waktu dan hari, (b) kondisi lingkungan (suhu, kelembapan, aktivitas dapur), dan (c) riwayat beban pada interval sebelumnya.

**Jenis tugas:** time-series regression.

**Kegunaan.**
- Baseline "beban wajar": bila beban aktual jauh di atas prediksi, bisa dicurigai ada perangkat yang lupa dimatikan (dasar deteksi anomali).
- Perencanaan kapasitas dan penjadwalan penggunaan listrik (*demand response*).
- Bahan perbandingan: seberapa jauh pola penggunaan di gedung ini bisa diprediksi dari kalender dan lingkungan.

**Metrik.**
- **RMSE** (utama, satuan VA, menghukum large error), **MAE** (error rata-rata, mudah dibaca), dan **R2** (porsi variansi yang dijelaskan).
- Pembanding wajib: *baseline* rata-rata dan *seasonal naive* ("beban sama dengan kemarin pada jam yang sama"). Model dianggap berguna hanya bila mengalahkan keduanya.

In [ ]:
print("hello world")

hello world


# **Load Dataset**

# **Spark Session Initialization**

In [3]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder as SparkOHE, VectorAssembler, StandardScaler as SparkScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor as SparkRF, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (SparkSession.builder.master("local[*]").appName("smart-building-ml")
         .config("spark.driver.memory", "2g")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| core:", spark.sparkContext.defaultParallelism)

# SCHEMA = StructType([StructField(c, DoubleType()) for c in NUM] +
#                     [StructField("dow", StringType()), StructField("y", DoubleType()), StructField("rid", IntegerType())])

# def to_spark(pdf):
#     # pandas -> Spark DataFrame (dibuat dari list agar tidak bergantung pada versi pandas/Arrow)
#     cols = [pdf[c].astype(float).tolist() for c in NUM]
#     rows = list(zip(*cols, pdf["dow"].astype(int).astype(str).tolist(), pdf["y"].astype(float).tolist(), range(len(pdf))))
#     return spark.createDataFrame(rows, SCHEMA)

# sdf_tr, sdf_va, sdf_te = to_spark(train).cache(), to_spark(val).cache(), to_spark(test).cache()
# print("Baris Spark (latih/validasi/uji):", sdf_tr.count(), sdf_va.count(), sdf_te.count())
# sdf_tr.printSchema()

Spark 4.0.4 | core: 2


In [ ]:
def spark_pipe(estimator, scale):
    stages = [StringIndexer(inputCol="dow", outputCol="dow_idx", handleInvalid="keep"),
              SparkOHE(inputCols=["dow_idx"], outputCols=["dow_ohe"], handleInvalid="keep"),
              VectorAssembler(inputCols=NUM + ["dow_ohe"], outputCol="features" if not scale else "feat_raw")]
    if scale:
        stages.append(SparkScaler(inputCol="feat_raw", outputCol="features", withMean=True, withStd=True))
    return SparkPipeline(stages=stages + [estimator])

SPARK_MODELS = {
    "Ridge": (lambda **p: LinearRegression(featuresCol="features", labelCol="y", elasticNetParam=0.0, **p),
              {"regParam": [0.001, 0.01, 0.1, 1.0]}, True),
    "RandomForest": (lambda **p: SparkRF(featuresCol="features", labelCol="y", seed=SEED, **p),
                     {"numTrees": [50, 100], "maxDepth": [4, 8], "minInstancesPerNode": [1, 3]}, False),
    "GradientBoosting": (lambda **p: GBTRegressor(featuresCol="features", labelCol="y", seed=SEED, **p),
                         {"maxIter": [30, 80], "stepSize": [0.1], "maxDepth": [2, 3]}, False),
}
ev = {m: RegressionEvaluator(labelCol="y", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}

# Pemanasan JVM agar waktu fit pertama tidak terinflasi oleh start-up Spark
spark_pipe(LinearRegression(featuresCol="features", labelCol="y"), True).fit(sdf_tr).transform(sdf_va).count()
print("Siap.")

In [ ]:
def spark_cv_rmse(make_est, params, scale):
    scores = []
    for tr_i, va_i in FOLDS:
        tr_df = sdf_tr.filter((F.col("rid") >= int(tr_i.min())) & (F.col("rid") <= int(tr_i.max())))
        va_df = sdf_tr.filter((F.col("rid") >= int(va_i.min())) & (F.col("rid") <= int(va_i.max())))
        model = spark_pipe(make_est(**params), scale).fit(tr_df)
        scores.append(ev["rmse"].evaluate(model.transform(va_df)))
    return float(np.mean(scores))

def spark_predict(model, sdf):
    return np.array([r["prediction"] for r in model.transform(sdf).orderBy("rid").select("prediction").collect()])

def spark_infer_ms(model, sdf, reps=5):
    t0 = time.perf_counter()
    for _ in range(reps):
        model.transform(sdf).select("prediction").collect()
    return 1000 * (time.perf_counter() - t0) / reps

SPARK_BEST = {}
for name, (make_est, grid, scale) in SPARK_MODELS.items():
    t0 = time.perf_counter()
    scored = [(spark_cv_rmse(make_est, p, scale), p) for p in ParameterGrid(grid)]
    tune_s = time.perf_counter() - t0
    cv_rmse, best_p = min(scored, key=lambda t: t[0])

    t0 = time.perf_counter()
    model = spark_pipe(make_est(**best_p), scale).fit(sdf_tr)
    fit_s = time.perf_counter() - t0
    SPARK_BEST[name] = model
    for nama, sdf, part in [("validasi", sdf_va, val), ("uji", sdf_te, test)]:
        p = spark_predict(model, sdf)
        PRED[(name, "spark", nama)] = p
        log(name, "spark", nama, part["y"].values, p, fit_s=fit_s, infer_ms=spark_infer_ms(model, sdf), params=str(best_p))
    print(f"{name:<17} CV-RMSE={cv_rmse:7.1f} | tuning {tune_s:5.1f}s ({len(scored) * len(FOLDS)} fit) | terbaik: {best_p}")